# Example: Output-COSMIC -- LTV Identification from Partial Observations

This example demonstrates the Output-COSMIC workflow for identifying
time-varying systems when only partial state measurements are available:

$$x(k+1) = A(k)\,x(k) + B(k)\,u(k) \quad \text{(unknown dynamics)}$$
$$y(k) = H\,x(k) \quad \text{(partial observation)}$$

The workflow is:
1. Estimate frequency response from input-output data
2. Determine model order (state dimension) via Hankel SVD
3. Construct observation matrix H
4. Identify LTV system matrices using `sid.ltv_disc_io`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sid

%matplotlib inline

## System Setup

A 4th-order LTI system with 2 measured outputs (`py = 2`, `n = 4`).

In [ ]:
rng = np.random.default_rng(42)
n = 4    # state dimension
q = 1    # single input
py = 2   # two measured outputs

# Stable dynamics: two pairs of complex poles
A_true = np.array([
    [0.8,  0.15,  0.0,  0.0],
    [-0.15, 0.8,  0.0,  0.0],
    [0.0,   0.0,  0.6,  0.2],
    [0.0,   0.0, -0.2,  0.6],
])

B_true = np.array([[1.0], [0.3], [0.5], [0.1]])

# Observation matrix: measure states 1 and 3
H_obs = np.array([
    [1, 0, 0, 0],
    [0, 0, 1, 0],
])

## Simulate Trajectories

In [ ]:
N = 80    # time steps per trajectory
L = 10    # number of trajectories
sigma_proc = 0.02   # process noise
sigma_meas = 0.05   # measurement noise

X = np.zeros((N + 1, n, L))
U = rng.standard_normal((N, q, L))
Y = np.zeros((N + 1, py, L))

for l in range(L):
    X[0, :, l] = 0.5 * rng.standard_normal(n)
    Y[0, :, l] = H_obs @ X[0, :, l] + sigma_meas * rng.standard_normal(py)
    for k in range(N):
        X[k + 1, :, l] = (A_true @ X[k, :, l]
                          + B_true.ravel() * U[k, 0, l]
                          + sigma_proc * rng.standard_normal(n))
        Y[k + 1, :, l] = H_obs @ X[k + 1, :, l] + sigma_meas * rng.standard_normal(py)

print(f'Simulated {L} trajectories of length {N}.')
print(f'True state dimension: n = {n}, observed: py = {py}.')

## Step 1: Estimate Frequency Response

Use the first trajectory for frequency response estimation.
Trim Y to match U length (`sid.freq_bt` requires equal-length signals).

In [ ]:
G = sid.freq_bt(Y[:N, :, 0], U[:, :, 0], window_size=20)

## Step 2: Model Order Determination

`sid.model_order` estimates n from the Hankel singular values.

In [ ]:
n_est, sv_info = sid.model_order(G)
print(f'Estimated model order: n = {n_est} (true = {n}).')

## Step 3: Construct Observation Matrix

In practice, H encodes which states are measured.
Here we use the true H since we know which channels correspond to which states.

In [ ]:
H_use = H_obs.copy()
print(f'Observation matrix H ({H_use.shape[0]} x {H_use.shape[1]}):\n{H_use}')

## Step 4: Identify LTV System from Partial Observations

Run Output-COSMIC identification via `sid.ltv_disc_io`.

In [ ]:
print('Running Output-COSMIC identification...')
result = sid.ltv_disc_io(Y, U, H_use, lambda_=1e5)

print(f'Converged in {result.iterations} iterations.')
print(f'Final cost: {result.cost[-1]:.4f}')

## Compare Recovered A to True A

In [ ]:
A_mean = result.a.mean(axis=2)
B_mean = result.b.mean(axis=2)
errA = np.linalg.norm(A_mean - A_true, 'fro') / np.linalg.norm(A_true, 'fro')
errB = np.linalg.norm(B_mean - B_true, 'fro') / np.linalg.norm(B_true, 'fro')
print(f'A recovery error (relative Frobenius): {errA:.4f}')
print(f'B recovery error (relative Frobenius): {errB:.4f}')

## Plot: Recovered A(1,1) Over Time vs True Value

In [ ]:
fig, ax = plt.subplots()
ax.plot(range(1, N + 1), result.a[0, 0, :], 'b-', label='Recovered $A_{11}(k)$')
ax.axhline(A_true[0, 0], color='k', linestyle='--', linewidth=1.5, label='True $A_{11}$')
ax.set_xlabel('Time step k')
ax.set_ylabel('$A_{11}(k)$')
ax.set_title('Output-COSMIC: Recovered Dynamics (LTI Case)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Plot: Estimated vs True States (Trajectory 1)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6))

# Observed state
axes[0].plot(np.arange(N + 1), X[:, 0, 0], 'k-', linewidth=1.5, label='True $x_1$')
axes[0].plot(np.arange(N + 1), result.x[:, 0, 0], 'b--', label='Estimated $x_1$')
axes[0].plot(np.arange(N + 1), Y[:, 0, 0], 'r.', markersize=4, label='Measurement $y_1$')
axes[0].legend()
axes[0].set_xlabel('k')
axes[0].set_ylabel('$x_1(k)$')
axes[0].set_title('State Recovery: Observed State')
axes[0].grid(True)

# Hidden state
axes[1].plot(np.arange(N + 1), X[:, 1, 0], 'k-', linewidth=1.5, label='True $x_2$')
axes[1].plot(np.arange(N + 1), result.x[:, 1, 0], 'b--', label='Estimated $x_2$')
axes[1].legend()
axes[1].set_xlabel('k')
axes[1].set_ylabel('$x_2(k)$')
axes[1].set_title('State Recovery: Hidden State')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Plot: Convergence History

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(np.arange(1, len(result.cost) + 1), result.cost, 'b-o', markersize=4)
ax.set_xlabel('Iteration')
ax.set_ylabel('Cost J')
ax.set_title('Output-COSMIC: Convergence')
ax.grid(True)
plt.tight_layout()
plt.show()

## Model Validation with `sid.compare` and `sid.residual`

Validate the identified model using the estimated states as reference.

In [ ]:
comp = sid.compare(result, result.x, U)
print('Model fit (per state component):')
for ch in range(n):
    print(f'  x_{ch + 1}: {comp["fit"][ch]:.1f}%')

# Residual diagnostics on estimated states
resid = sid.residual(result, result.x, U)
if resid['whiteness_pass']:
    print('Residual whiteness test: PASS')
else:
    print('Residual whiteness test: FAIL (model may need refinement)')

# Observation-space check: H @ X_hat should approximate Y
Y_recon = np.zeros((N + 1, py, L))
for l in range(L):
    Y_recon[:, :, l] = result.x[:, :, l] @ H_use.T
obs_err = np.linalg.norm(Y_recon.ravel() - Y.ravel()) / np.linalg.norm(Y.ravel())
print(f'Observation reconstruction error: {obs_err:.4f}')

## Frozen Transfer Function

Compute the frozen transfer function at the midpoint and inspect the result.

In [ ]:
# Build a lightweight LTV result at the midpoint for frozen TF
mid_k = N // 2
print(f'A at midpoint (k={mid_k + 1}):\n{result.a[:, :, mid_k]}')
print(f'B at midpoint (k={mid_k + 1}):\n{result.b[:, :, mid_k]}')

print('\nExample completed successfully.')